In [4]:
! pip install pysmt

In [6]:
from pysmt.smtlib.parser import SmtLibParser

parser = SmtLibParser()

with open(r"E:\smt_benchmark\QF_BV\non-incremental\QF_BV\bench_ab\a49test0010.smt2", "r", encoding="utf-8") as f:
    script = parser.get_script(f)

print(script)

SmtLibCommand(name='set-info', args=[':smt-lib-version', '2.6'])
SmtLibCommand(name='set-logic', args=[QF_BV])
SmtLibCommand(name='set-info', args=[':source', "\nBit-vector benchmarks from Dawson Engler's tool contributed by Vijay Ganesh\n(vganesh@stanford.edu).  Translated into SMT-LIB format by Clark Barrett using\nCVC3.\n\n"])
SmtLibCommand(name='set-info', args=[':category', '"industrial"'])
SmtLibCommand(name='set-info', args=[':status', 'sat'])
SmtLibCommand(name='declare-fun', args=[a])
SmtLibCommand(name='declare-fun', args=[b])
SmtLibCommand(name='assert', args=[((a & 16_32) = 0_32)])
SmtLibCommand(name='assert', args=[(! (a s< (b >> 1_32)))])
SmtLibCommand(name='assert', args=[(! ((a & 8_32) = 0_32))])
SmtLibCommand(name='assert', args=[((a & 4_32) = 0_32)])
SmtLibCommand(name='check-sat', args=[])
SmtLibCommand(name='exit', args=[])


In [7]:
for cmd in script.commands:
    print(cmd.name, cmd.args)

set-info [':smt-lib-version', '2.6']
set-logic [QF_BV]
set-info [':source', "\nBit-vector benchmarks from Dawson Engler's tool contributed by Vijay Ganesh\n(vganesh@stanford.edu).  Translated into SMT-LIB format by Clark Barrett using\nCVC3.\n\n"]
set-info [':category', '"industrial"']
set-info [':status', 'sat']
declare-fun [a]
declare-fun [b]
assert [((a & 16_32) = 0_32)]
assert [(! (a s< (b >> 1_32)))]
assert [(! ((a & 8_32) = 0_32))]
assert [((a & 4_32) = 0_32)]
check-sat []
exit []


In [10]:
from z3 import *

def solve_smt2_file(path: str):
    s = Solver()

    # 读取 .smt2 文件里的断言
    s.from_file(path)

    result = s.check()
    print("result =", result)

    if result == sat:
        m = s.model()
        print("model:")
        print(m)

    elif result == unsat:
        print("UNSAT")

    else:
        print("UNKNOWN")


if __name__ == "__main__":
    solve_smt2_file(r"E:\smt_benchmark\QF_BV\non-incremental\QF_BV\bench_ab\a49test0010.smt2")

result = sat
model:
[a = 2147467114, b = 0]


In [9]:
from pysmt.smtlib.parser import SmtLibParser
from pysmt.typing import BVType, BOOL, INT

def convert_sort(sort):
    if sort.is_bv_type():
        return {"kind": "BitVec", "width": sort.width}
    if sort.is_bool_type():
        return {"kind": "Bool"}
    if sort.is_int_type():
        return {"kind": "Int"}
    return {"kind": "UnknownSort", "repr": str(sort)}

def convert_term(term):
    if term.is_symbol():
        return {
            "kind": "symbol",
            "name": term.symbol_name(),
            "sort": convert_sort(term.get_type())
        }

    if term.is_bv_constant():
        return {
            "kind": "bvconst",
            "value": term.bv_unsigned_value(),
            "width": term.bv_width()
        }

    if term.is_bool_constant():
        return {
            "kind": "boolconst",
            "value": term.constant_value()
        }

    if term.is_int_constant():
        return {
            "kind": "intconst",
            "value": term.constant_value()
        }

    return {
        "kind": "app",
        "op": str(term.node_type()),
        "repr": str(term),
        "args": [convert_term(a) for a in term.args()],
        "sort": convert_sort(term.get_type())
    }

def convert_command(cmd):
    name = cmd.name

    if name == "set-logic":
        return {"type": "set-logic", "logic": cmd.args[0]}

    if name == "set-info":
        return {"type": "set-info", "key": cmd.args[0], "value": cmd.args[1]}

    if name == "declare-fun":
        fname, arg_sorts, ret_sort = cmd.args
        return {
            "type": "declare-fun",
            "name": fname,
            "args": [convert_sort(s) for s in arg_sorts],
            "return_sort": convert_sort(ret_sort)
        }

    if name == "assert":
        return {
            "type": "assert",
            "term": convert_term(cmd.args[0])
        }

    if name == "check-sat":
        return {"type": "check-sat"}

    return {
        "type": name,
        "raw_args": [str(x) for x in cmd.args]
    }

parser = SmtLibParser()
with open(r"E:\smt_benchmark\QF_BV\non-incremental\QF_BV\bench_ab\a49test0010.smt2", "r", encoding="utf-8") as f:
    script = parser.get_script(f)

ast = {
    "type": "script",
    "commands": [convert_command(cmd) for cmd in script.commands]
}

import json
print(json.dumps(ast, indent=2, ensure_ascii=False))

ValueError: not enough values to unpack (expected 3, got 1)